# 3D mapping with deck.gl: pickups, facilities, and height as data

This notebook embeds a browser-native deck.gl scene. It uses extruded columns to encode taxi fare and facility locations as contrasting points.

The example is intentionally JavaScript-rendered so it can run in JupyterLite without a native `pydeck` install.

For deeper context, deck.gl is a WebGL/WebGPU visualization framework for large geospatial datasets.

**Reflection questions:** When does height improve understanding, and when does it distort perception? What should a legend communicate in a 3D map? How would you validate a 3D visualization for accessibility?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
taxi = load_csv('nyc_taxi_pickups_tiny.csv')
fac = load_csv('health_facilities_training_points.csv')
nyc_fac = fac[fac.city.eq('New York')]
from IPython.display import HTML
html_doc = f'''
<div id="deck-map" style="height:650px;"></div>
<script src="https://unpkg.com/deck.gl@9.0.0/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet"/>
<script>
const taxi = {json.dumps(taxi.to_dict(orient='records'))};
const facilities = {json.dumps(nyc_fac.to_dict(orient='records'))};
const deckgl = new deck.DeckGL({{
  container: 'deck-map',
  mapStyle: 'https://basemaps.cartocdn.com/gl/positron-gl-style/style.json',
  initialViewState: {{longitude: -73.96, latitude: 40.74, zoom: 10.5, pitch: 55, bearing: -20}},
  controller: true,
  layers: [
    new deck.ColumnLayer({{
      id: 'fares', data: taxi, diskResolution: 24, radius: 120, extruded: true, pickable: true,
      getPosition: d => [d.lon, d.lat], getElevation: d => d.fare_usd * 120, getFillColor: d => [255, 140, 0, 160]
    }}),
    new deck.ScatterplotLayer({{
      id: 'facilities', data: facilities, pickable: true, radiusMinPixels: 6, getRadius: 220,
      getPosition: d => [d.lon, d.lat], getFillColor: [20, 90, 220, 210]
    }})
  ],
  getTooltip: (info) => info.object && (info.object.pickup_zone ? `${{info.object.pickup_zone}} fare $${{info.object.fare_usd}}` : info.object.name)
}});
</script>
'''
HTML(html_doc)